In [ ]:
# Install required libraries
!pip install -q datasets transformers torch

In [1]:


import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer
import numpy as np
import time
import os

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"GPU Name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

Using device: cuda
GPU Name: Tesla T4


In [13]:
print("Loading TinyStories dataset...")
# Load a small subset for quick Colab training
dataset = load_dataset("roneneldan/TinyStories", split="train[:10000]")

print("Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=64, padding="max_length")

print("Tokenizing...")
tokenized_datasets = dataset.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_datasets.set_format(type="torch", columns=["input_ids", "attention_mask"])

# Create a simple DataLoader
from torch.utils.data import DataLoader
dataloader = DataLoader(tokenized_datasets, batch_size=16, shuffle=True)

print(f"Dataset ready! Total batches: {len(dataloader)}")

Loading TinyStories dataset...
Loading Tokenizer...
Tokenizing...


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Dataset ready! Total batches: 625


In [14]:
class SparseMoEFFN(nn.Module):
    def __init__(self, d_model, n_experts=4, top_k=2):
        super().__init__()
        self.n_experts = n_experts
        self.top_k = top_k
        self.gate = nn.Linear(d_model, n_experts)

        # Experts are simple 2-layer networks
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, d_model * 2),
                nn.GELU(),
                nn.Linear(d_model * 2, d_model)
            )
            for _ in range(n_experts)
        ])

    def forward(self, x):
        # x shape: (batch, seq_len, d_model)
        batch, seq_len, d_model = x.shape

        # 1. Routing: Calculate probability of sending token to each expert
        router_logits = self.gate(x) # (batch, seq_len, n_experts)
        router_probs = F.softmax(router_logits, dim=-1)

        # 2. Top-K Selection: Keep only the top_k experts per token
        topk_probs, topk_indices = torch.topk(router_probs, self.top_k, dim=-1)

        # Create a sparse mask (zero out non-selected experts)
        mask = torch.zeros_like(router_probs).scatter_(-1, topk_indices, 1.0)
        sparse_router_probs = router_probs * mask

        # Renormalize so the top_k probabilities sum to 1
        sparse_router_probs = sparse_router_probs / (sparse_router_probs.sum(dim=-1, keepdim=True) + 1e-9)

        # 3. Compute Experts (Vectorized for Colab stability)
        # expert_outputs: (n_experts, batch, seq_len, d_model)
        expert_outputs = torch.stack([expert(x) for expert in self.experts], dim=0)

        # Multiply expert outputs by their routing weights
        # sparse_router_probs permuted: (n_experts, batch, seq_len, 1)
        weights = sparse_router_probs.permute(2, 0, 1).unsqueeze(-1)
        weighted_expert_out = expert_outputs * weights

        # Sum across the expert dimension to get final output
        final_output = weighted_expert_out.sum(dim=0)

        # 4. Load Balancing Loss (Crucial for MoE!)
        # We want all experts to be used equally. We penalize deviation from uniform distribution.
        mean_probs = router_probs.mean(dim=(0, 1)) # Average probability per expert
        uniform_target = torch.ones_like(mean_probs) / self.n_experts
        load_balance_loss = F.mse_loss(mean_probs, uniform_target)

        return final_output, load_balance_loss

class ASTN_Block(nn.Module):
    """A single Post-Transformer Block with Sparse MoE"""
    def __init__(self, d_model, n_heads, n_experts=4, top_k=2):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.moe_ffn = SparseMoEFFN(d_model, n_experts, top_k)

    # Renamed attn_mask to key_padding_mask for clarity and added causal_mask
    def forward(self, x, key_padding_mask=None, causal_mask=None):
        # Self-Attention
        # Pass key_padding_mask to `key_padding_mask` arg and causal_mask to `attn_mask` arg
        attn_out, _ = self.attn(x, x, x, key_padding_mask=key_padding_mask, attn_mask=causal_mask)
        x = self.ln1(x + attn_out)

        # Sparse MoE Feed Forward
        ffn_out, lb_loss = self.moe_ffn(x)
        x = self.ln2(x + ffn_out)

        return x, lb_loss

class TinyASTN(nn.Module):
    def __init__(self, vocab_size, d_model=128, n_heads=4, n_layers=2, n_experts=4, top_k=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Parameter(torch.zeros(1, 64, d_model)) # Max seq len 64

        self.blocks = nn.ModuleList([
            ASTN_Block(d_model, n_heads, n_experts, top_k) for _ in range(n_layers)
        ])

        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        self.d_model = d_model
        # Store max_seq_len for causal mask generation
        self.max_seq_len = 64

    # Renamed attention_mask to key_padding_mask for clarity
    def forward(self, input_ids, key_padding_mask=None):
        seq_len = input_ids.size(1)
        # Slice pos_embed to match the current sequence length, ensuring the latest definition is used
        x = self.embed(input_ids) + self.pos_embed[:, :seq_len, :]

        total_lb_loss = 0.0

        # Create causal mask (look-ahead mask) for decoder-style attention
        # `generate_square_subsequent_mask` returns a boolean mask where `True` indicates masked positions.
        causal_mask = nn.Transformer.generate_square_subsequent_mask(seq_len).to(x.device)

        for block in self.blocks:
            # Pass both key_padding_mask and causal_mask
            x, lb_loss = block(x, key_padding_mask=key_padding_mask, causal_mask=causal_mask)
            total_lb_loss += lb_loss

        x = self.ln_f(x)
        logits = self.lm_head(x)

        return logits, total_lb_loss

In [15]:
# Initialize Model
vocab_size = tokenizer.vocab_size
model = TinyASTN(vocab_size=vocab_size, d_model=128, n_heads=4, n_layers=2, n_experts=4, top_k=2).to(device)

# Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)

# Training Settings
epochs = 10
print("Starting Pre-training...")

for epoch in range(epochs):
    model.train()
    total_loss = 0
    start_time = time.time()

    for batch_idx, batch in enumerate(dataloader):
        input_ids = batch['input_ids'].to(device)
        # Convert attention_mask to a boolean mask for key_padding_mask
        # True means masked (ignored), so mask out positions where original attention_mask is 0 (padding)
        key_padding_mask = (batch['attention_mask'] == 0).to(device)
        labels = input_ids.clone()

        optimizer.zero_grad()

        # Forward pass, pass the converted mask as key_padding_mask
        logits, lb_loss = model(input_ids, key_padding_mask=key_padding_mask)

        # Shift logits and labels for next-token prediction
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()

        # Cross Entropy Loss
        ce_loss = F.cross_entropy(shift_logits.view(-1, vocab_size), shift_labels.view(-1), ignore_index=tokenizer.pad_token_id)

        # Total Loss = CE Loss + (Weight * Load Balance Loss)
        # We weight LB loss lightly so it guides routing without dominating the language objective
        loss = ce_loss + (0.05 * lb_loss)

        loss.backward()

        # Gradient clipping for stability
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        total_loss += loss.item()

        if batch_idx % 50 == 0:
            print(f"Epoch {epoch+1} | Batch {batch_idx}/{len(dataloader)} | Loss: {loss.item():.4f} | CE: {ce_loss.item():.4f} | LB: {lb_loss.item():.4f}")

    avg_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch+1} Completed. Average Loss: {avg_loss:.4f} | Time: {time.time() - start_time:.2f}s")

# SAVE THE MODEL
torch.save(model.state_dict(), "tiny_astn_trained.pth")
print("✅ Model saved to 'tiny_astn_trained.pth'")

Starting Pre-training...
Epoch 1 | Batch 0/625 | Loss: 10.9654 | CE: 10.9653 | LB: 0.0013


/usr/local/lib/python3.13/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(


Epoch 1 | Batch 50/625 | Loss: 7.6380 | CE: 7.6379 | LB: 0.0016
Epoch 1 | Batch 100/625 | Loss: 5.2962 | CE: 5.2961 | LB: 0.0017
Epoch 1 | Batch 150/625 | Loss: 4.9921 | CE: 4.9920 | LB: 0.0012
Epoch 1 | Batch 200/625 | Loss: 4.9260 | CE: 4.9259 | LB: 0.0023
Epoch 1 | Batch 250/625 | Loss: 4.5027 | CE: 4.5026 | LB: 0.0027
Epoch 1 | Batch 300/625 | Loss: 4.0503 | CE: 4.0502 | LB: 0.0017
Epoch 1 | Batch 350/625 | Loss: 4.2077 | CE: 4.2076 | LB: 0.0022
Epoch 1 | Batch 400/625 | Loss: 4.1255 | CE: 4.1253 | LB: 0.0021
Epoch 1 | Batch 450/625 | Loss: 3.6081 | CE: 3.6080 | LB: 0.0013
Epoch 1 | Batch 500/625 | Loss: 3.9234 | CE: 3.9233 | LB: 0.0008
Epoch 1 | Batch 550/625 | Loss: 3.4696 | CE: 3.4696 | LB: 0.0018
Epoch 1 | Batch 600/625 | Loss: 3.8876 | CE: 3.8876 | LB: 0.0010
Epoch 1 Completed. Average Loss: 4.7504 | Time: 21.20s
Epoch 2 | Batch 0/625 | Loss: 3.4622 | CE: 3.4621 | LB: 0.0011
Epoch 2 | Batch 50/625 | Loss: 3.4674 | CE: 3.4674 | LB: 0.0005
Epoch 2 | Batch 100/625 | Loss: 3.5108 

In [16]:
# 1. Load the saved model
print("Loading trained model...")
loaded_model = TinyASTN(vocab_size=vocab_size, d_model=128, n_heads=4, n_layers=2, n_experts=4, top_k=2).to(device)
loaded_model.load_state_dict(torch.load("tiny_astn_trained.pth", map_location=device))
loaded_model.eval()



# 2. Generation Function
def generate_story(prompt, max_length=60, temperature=0.8, top_k=50):
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

    generated = input_ids.clone()

    with torch.no_grad():
        for _ in range(max_length):
            # Get model predictions
            # Explicitly pass key_padding_mask=None during generation
            logits, _ = loaded_model(generated, key_padding_mask=None)

            # Get logits for the last token
            next_token_logits = logits[:, -1, :] / temperature

            # Top-K filtering
            top_k_logits, top_k_indices = torch.topk(next_token_logits, top_k)
            next_token_logits = torch.full_like(next_token_logits, float('-inf'))
            next_token_logits.scatter_(1, top_k_indices, top_k_logits)

            # Sample
            probs = F.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)

            # Append to generated sequence
            generated = torch.cat((generated, next_token), dim=1)

            # Stop if EOS token is generated
            if next_token.item() == tokenizer.eos_token_id:
                break

    return tokenizer.decode(generated[0], skip_special_tokens=True)


Loading trained model...


In [17]:
# 3. Test Inference
print("\n--- Generating TinyStories ---")
prompts = [
    "Once upon a time, there was a little ",
    "The cat wanted to ",
    "Tommy found a magic "
]

for p in prompts:
    print(f"Prompt: {p}")
    story = generate_story(p, max_length=40, temperature=0.7)
    print(f"Generated: {story}\n")


--- Generating TinyStories ---
Prompt: Once upon a time, there was a little 
Generated: Once upon a time, there was a little y cat named Fluffy. Fluffy loved to play with his friends. One day, Fluffy saw a big brown dog with a big, shiny red ball. The dog was very tall and shiny

Prompt: The cat wanted to 
Generated: The cat wanted to 
"What's.
"Please help me?" asked his mom said it.

"That's a little girl," said.

"That's so excited to get her hands.

Prompt: Tommy found a magic 
Generated: Tommy found a magic  was very impatient. He wanted to show his mommy and dad to it.

Mia's dad gave him a big and a hat for his own. It was a lot of fun toys



In [19]:
!gred | 'tiny_astn_trained.pth'

/bin/bash: line 1: gred: command not found
/bin/bash: line 1: tiny_astn_trained.pth: command not found


In [20]:
!ls -a

.  ..  .config	sample_data  tiny_astn_trained.pth


In [ ]:
!cp tiny_astn_trained.pth